# Build a Content Agent

Build a multi-step agent that analyzes video footage and produces structured creative output using domain instructions, structured output, and multi-turn refinement.

# Install the TwelveLabs Python SDK

In [ ]:
%pip install twelvelabs

In [ ]:
import json
import os

from twelvelabs import TwelveLabs, IngestionConfig, EnrichmentConfig_Description, TextParam
from twelvelabs.types.text_param_format import TextParamFormat_JsonSchema

# Configuration
API_KEY = os.environ.get("TWELVELABS_API_KEY", "YOUR_API_KEY")

client = TwelveLabs(api_key=API_KEY)

## Helper Functions

Utility functions for parsing Jockey API responses.

In [ ]:
def parse_response(response) -> str:
    """Extract text content from a Jockey response.

    Args:
        response: The ResponseObject returned by client.responses.create().

    Returns:
        The text content from the first message output, or an empty string
        if no message content is found.
    """
    for output in response.output:
        if output.type == "message":
            for content in output.content:
                return content.text
    return ""


def parse_json_response(response) -> dict:
    """Extract and parse JSON content from a Jockey response."""
    text = parse_response(response)
    if text:
        return json.loads(text)
    return {}

## Step 1: Create a Domain-Focused Knowledge Store

Tailor the ingestion config to what your agent needs to extract. The `enrichment_config.description` field tells the indexer what to focus on when processing videos.

For a micro-drama agent, we focus on characters, emotions, interpersonal dynamics, conflicts, and dramatic turning points.

In [ ]:
INGESTION_DESCRIPTION = (
    "Focus on characters and their emotions, interpersonal dynamics, "
    "conflicts, tension points, visual mood shifts, dialogue tone, "
    "and dramatic turning points. Track recurring characters across videos."
)

store = client.knowledge_stores.create(
    name="Microdrama Source Material",
    ingestion_config=IngestionConfig(
        enrichment_config=EnrichmentConfig_Description(
            description=INGESTION_DESCRIPTION
        )
    ),
)

store_id = store.id
print(f"Created knowledge store: {store_id}")

## Step 2: Define the Drama Schema

The schema captures the full micro-drama outline: title, logline, character list with arcs, scenes with video references and timestamps, central conflict, and resolution. This gives you typed, parseable output from each query.

In [ ]:
DRAMA_SCHEMA = {
    "type": "object",
    "properties": {
        "title": {"type": "string"},
        "logline": {"type": "string"},
        "characters": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "name": {"type": "string"},
                    "role": {"type": "string"},
                    "arc": {"type": "string"},
                },
            },
        },
        "scenes": {
            "type": "array",
            "items": {
                "type": "object",
                "properties": {
                    "scene_number": {"type": "integer"},
                    "description": {"type": "string"},
                    "video_reference": {"type": "string"},
                    "timestamp": {"type": "string"},
                    "dramatic_function": {"type": "string"},
                },
            },
        },
        "central_conflict": {"type": "string"},
        "resolution": {"type": "string"},
    },
}

## Step 3: Query with Domain Instructions + Structured Output

Combine three features in a single request:

1. **Domain instructions** -- specialize the agent's behavior at query time ("You are a micro-drama creator...").
2. **Structured output** -- enforce the drama schema so results are typed and parseable.
3. **Knowledge store tool** -- give the agent access to your indexed video collection.

The `session_id` returned in the response enables multi-turn refinement in the next step.

In [ ]:
response = client.responses.create(
    knowledge_store_id=store_id,
    instructions=(
        "You are a micro-drama creator. Analyze the video collection "
        "for narrative potential. Identify characters, conflicts, and "
        "dramatic moments. Construct a compelling 60-second micro-drama "
        "outline."
    ),
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "Create a micro-drama from these videos. "
                "Focus on the strongest emotional arc you can find."
            ),
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="micro_drama", schema_=DRAMA_SCHEMA)
    ),
)

session_id = response.session_id
drama = parse_json_response(response)

print(f"Title: {drama['title']}")
print(f"Logline: {drama['logline']}")
print(f"Central Conflict: {drama['central_conflict']}")
print(f"Resolution: {drama['resolution']}")
print()
print("Characters:")
for char in drama["characters"]:
    print(f"  {char['name']} ({char['role']}): {char['arc']}")
print()
print("Scenes:")
for scene in drama["scenes"]:
    print(f"  Scene {scene['scene_number']}: {scene['description']}")
    print(f"    Video: {scene['video_reference']} @ {scene['timestamp']}")
    print(f"    Function: {scene['dramatic_function']}")

## Step 4: Refine in Multi-Turn

Use the `session_id` to continue the conversation. Each follow-up turn has access to the full conversation history, so Jockey can refine its output based on prior context.

Common refinement patterns:
- Sharpen the conflict or turning point
- Swap a scene for a better clip
- Adjust pacing or tone
- Extend the drama into multiple episodes

In [ ]:
response = client.responses.create(
    knowledge_store_id=store_id,
    session_id=session_id,
    input=[
        {
            "type": "message",
            "role": "user",
            "content": (
                "Make the conflict sharper. "
                "Can you find a stronger turning point moment?"
            ),
        }
    ],
    text=TextParam(
        format=TextParamFormat_JsonSchema(name="micro_drama", schema_=DRAMA_SCHEMA)
    ),
)

refined_drama = parse_json_response(response)

print(f"Refined Title: {refined_drama.get('title', 'N/A')}")
print(f"Refined Conflict: {refined_drama.get('central_conflict', 'N/A')}")
print()
for scene in refined_drama.get("scenes", []):
    print(f"  Scene {scene['scene_number']}: {scene['description']}")

## The Pattern

This cookbook demonstrates a reusable agent pattern with four components:

1. **Domain ingestion config** -- Tell Jockey what to focus on during indexing.
2. **Domain instructions** -- Specialize the query-time behavior.
3. **Structured output** -- Get typed, parseable results.
4. **Multi-turn refinement** -- Iterate toward the desired output.

Swap the domain to build different agents:

| Agent Type | Ingestion Focus | Instructions | Output Schema |
|-----------|----------------|-------------|---------------|
| Micro-drama | Characters, emotions, conflicts | "You are a micro-drama creator..." | Scenes, characters, arcs |
| Training curriculum | Learning objectives, demonstrations | "You are an instructional designer..." | Modules, objectives, assessments |
| Compliance auditor | Claims, disclosures, regulations | "You are a compliance reviewer..." | Findings, severity, recommendations |
| Content planner | Topics, audience signals, gaps | "You are a content strategist..." | Calendar, themes, priorities |

## Variations

- **Genre shift:** Change instructions to "horror micro-drama" or "comedy sketch" or "documentary short".
- **Character focus:** "Build the drama around the most frequently appearing person."
- **Multi-episode:** Use session continuity to create a series: "Now create episode 2 continuing from this ending."

## Next Steps

- [Organize a Video Library](./organize_video_library.ipynb) -- Categorize your video collection for better agent input.
- [Assemble Highlight Reels](./assemble_highlight_reels.ipynb) -- Find and assemble clips from your collection.
- [Build a Content Agent (docs)](https://docs.twelvelabs.io/v1.3/agents/recipes/build-a-content-agent) -- Full reference documentation.
- [Ingestion Config Guide](https://docs.twelvelabs.io/v1.3/agents/guides/create-a-knowledge-store/configure-ingestion) -- Learn how to tune ingestion config for your domain.